# ចម្លើយលំហាត់ ជំពូក ៤

> **ចំណាំអំពីប្រភព និងអាជ្ញាបណ្ណ** — នេះជា *companion notebook* របស់សហគមន៍ខ្មែរ។
> កូដទាំងអស់ក្នុង notebook នេះគឺ **ដូចគ្នា ១០០%** ទៅនឹង notebook ផ្លូវការរបស់ឃ្លាំង
> [rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch04/01_main-chapter-code/exercise-solutions.ipynb) (អាជ្ញាបណ្ណ Apache-2.0, (c) Sebastian Raschka)។
> ចំណែកអត្ថបទពន្យល់ជាភាសាខ្មែរគឺជាអត្ថបទដើមរបស់សហគមន៍ (CC BY 4.0) —
> **មិនមែន**ជាការបកប្រែអត្ថបទក្នុងសៀវភៅ *Build a Large Language Model (From Scratch)* ទេ
> ព្រោះអត្ថបទសៀវភៅជាកម្មសិទ្ធិរបស់ Manning Publications និងស្ថិតក្រៅវិសាលភាពអាជ្ញាបណ្ណបើកចំហ។
> សម្រាប់ការពន្យល់ពេញលេញ សូមអានសៀវភៅផ្លូវការ។

**កូដ៖** តក្កវិទ្យាកូដ (logic) ដូច upstream ទាំងស្រុង — មានតែ *comments* ក្នុងកូដដែលត្រូវបានបកប្រែជាភាសាខ្មែរដោយសហគមន៍; កូដដែល comment ជាប់ (commented-out code) នៅដដែល។

**របៀបដំណើរការ៖** ដាក់ notebook នេះក្នុងថតជំពូកពាក់ព័ន្របស់ឃ្លាំង upstream
(ឬចម្លងឯកសារជំនួយដែលកូដត្រូវការ ដូចជា `previous_chapters.py`, `gpt_download.py`
និងឯកសារទិន្នន័យ) រួចដំឡើង dependencies តាម `requirements.txt` ឬប្រើ Google Colab។

## មាតិកា

- ## ០. ការត្រៀម
- ## ១. ការរាប់ parameters ក្នុងប្លុក feed-forward និង attention
- ## ២. ការកែប្រែ config របស់ម៉ូដែល
- ## ៣. ការកែប្រែបន្ថែមលើស្រទាប់

## ០. ការត្រៀម

នាំចូលម៉ូឌុលពីកូដជំពូក ៤។

In [ ]:
from importlib.metadata import version

print("torch version:", version("torch"))

## ១. ការរាប់ parameters ក្នុងប្លុក feed-forward និង attention

គណនាចំនួន parameters របស់ប្លុករងនីមួយៗដាច់ដោយឡែក។

In [ ]:
from gpt import TransformerBlock

GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

block = TransformerBlock(GPT_CONFIG_124M)
print(block)

In [ ]:
total_params = sum(p.numel() for p in block.ff.parameters())
print(f"Total number of parameters in feed forward module: {total_params:,}")

In [ ]:
total_params = sum(p.numel() for p in block.att.parameters())
print(f"Total number of parameters in attention module: {total_params:,}")

## ២. ការកែប្រែ config របស់ម៉ូដែល

កែតម្លៃ config ហើយសាងសង់ម៉ូដែលថ្មីដើម្បីសង្កេតឥទ្ធិពល។

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}


def get_config(base_config, model_name="gpt2-small"):
    GPT_CONFIG = base_config.copy()

    if model_name == "gpt2-small":
        GPT_CONFIG["emb_dim"] = 768
        GPT_CONFIG["n_layers"] = 12
        GPT_CONFIG["n_heads"] = 12

    elif model_name == "gpt2-medium":
        GPT_CONFIG["emb_dim"] = 1024
        GPT_CONFIG["n_layers"] = 24
        GPT_CONFIG["n_heads"] = 16

    elif model_name == "gpt2-large":
        GPT_CONFIG["emb_dim"] = 1280
        GPT_CONFIG["n_layers"] = 36
        GPT_CONFIG["n_heads"] = 20

    elif model_name == "gpt2-xl":
        GPT_CONFIG["emb_dim"] = 1600
        GPT_CONFIG["n_layers"] = 48
        GPT_CONFIG["n_heads"] = 25

    else:
        raise ValueError(f"Incorrect model name {model_name}")

    return GPT_CONFIG


def calculate_size(model):  # ផ្អែកលើកូដជំពូក
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total number of parameters: {total_params:,}")

    total_params_gpt2 =  total_params - sum(p.numel() for p in model.out_head.parameters())
    print(f"Number of trainable parameters considering weight tying: {total_params_gpt2:,}")
    
    # គណនាទំហំសរុបជា bytes (សន្មត float32, 4 bytes ក្នុងមួយ parameter)
    total_size_bytes = total_params * 4
    
    # បំលែងទៅជា megabytes
    total_size_mb = total_size_bytes / (1024 * 1024)
    
    print(f"Total size of the model: {total_size_mb:.2f} MB")

In [ ]:
from gpt import GPTModel


for model_abbrev in ("small", "medium", "large", "xl"):
    model_name = f"gpt2-{model_abbrev}"
    CONFIG = get_config(GPT_CONFIG_124M, model_name=model_name)
    model = GPTModel(CONFIG)
    print(f"\n\n{model_name}:")
    calculate_size(model)

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate_emb": 0.1,  # ថ្មី៖ dropout សម្រាប់ស្រទាប់ embedding
    "drop_rate_attn": 0.1,  # ថ្មី៖ dropout សម្រាប់ multi-head attention
    "drop_rate_shortcut": 0.1,  # ថ្មី dropout សម្រាប់ការតភ្ជាប់កាត់
    "qkv_bias": False
}

## ៣. ការកែប្រែបន្ថែមលើស្រទាប់

កែស្រទាប់បន្ថែមដើម្បីបំពេញលក្ខខណ្ឌលំហាត់។

In [ ]:
import torch.nn as nn
from gpt import MultiHeadAttention, LayerNorm, FeedForward


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"], 
            dropout=cfg["drop_rate_attn"],  # ថ្មី៖ dropout សម្រាប់ multi-head attention
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate_shortcut"])

    def forward(self, x):
        # ការតភ្ជាប់កាត់ (shortcut) សម្រាប់ប្លុក attention
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)  # រូបរាង [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)
        x = x + shortcut  # ដាក់បញ្ចូលដើមត្រឡប់មកវិញ

        # ការតភ្ជាប់កាត់ (shortcut) សម្រាប់ប្លុក feed-forward
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # ដាក់បញ្ចូលដើមត្រឡប់មកវិញ

        return x


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate_emb"])  # ថ្មី៖ dropout សម្រាប់ស្រទាប់ embedding

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds  # រូបរាង [batch_size, num_tokens, emb_size]
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [ ]:
import torch

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)

---

## អាជ្ញាបណ្ណសង្ខេប

- កូដ៖ Apache-2.0 — (c) Sebastian Raschka, គម្រោង [LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch) (មិនបានកែប្រែ)
- អត្ថបទពន្យល់ខ្មែរ៖ CC BY 4.0 — សហគមន៍ឯកសារខ្មែរ
- ឯកសារសៀវភៅ និងរូបភាពពាក់ព័ន្ធ៖ កម្មសិទ្ធិ Manning Publications — មិនត្រូវបានចម្លងមកទីនេះទេ